In [3]:
import speech_recognition as sr

In [4]:
import pandas as pd
df = pd.read_csv("question_set.csv",encoding = 'ISO-8859-1')
df.head(1)


FileNotFoundError: [Errno 2] No such file or directory: 'question_set.csv'

In [5]:
df.isnull().sum()

NameError: name 'df' is not defined

In [ ]:
df.fillna("Not Mentioned",inplace = True)

C:\Users\amans\AppData\Local\Temp\ipykernel_14560\468159929.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Not Mentioned' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("Not Mentioned",inplace = True)


In [ ]:
df.isnull().sum()

sr_no       0
question    0
ans         0
ans1        0
dtype: int64

In [ ]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
def preprocess_text(text):
    text = text.lower()
    words = word_tokenize(text)
    stop_word = set(stopwords.words('english'))
    words = [i for i in words if i not in stop_word]
    stemmer = PorterStemmer()
    words = [stemmer.stem(i) for i in words]
    proprocessed_text = ' '.join(words)
    return proprocessed_text

In [ ]:
df['ans1'] = df['ans1'].apply(preprocess_text)
df.head(1)

,sr_no,question,ans,ans1
0,1.0,What is Python? List some popular applications...,"Python is a widely-used general-purpose, high-...","python widely-us general-purpos , high-level p..."


In [ ]:
import pickle
with open('preprocess_text.pkl','wb')as file:
    pickle.dump(df,file)

In [ ]:
df['question'] = df['question'].str.strip()

In [ ]:
df['question']

0    What is Python? List some popular applications...
1    What are the benefits of using Python language...
2    What is the difference between a Mutable datat...
3    What is the difference between a Set and Dicti...
4                            What is a pass in Python?
5    Difference between for loop and while loop in ...
6                                        Not Mentioned
Name: question, dtype: object

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer,util
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')



C:\Users\amans\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
def suggest_section_for_selected_question(ans, df, question_index):
    # Preprocess the user's answer
    preprocessed_ans = preprocess_text(ans)
    ans_embedding = model.encode(preprocessed_ans)

    # Get the actual answer for the selected question
    actual_ans = df.iloc[question_index]['ans']
    preprocessed_actual_ans = preprocess_text(actual_ans)
    actual_ans_embedding = model.encode(preprocessed_actual_ans)

    # Compute the similarity between the user's answer and the actual answer
    similarity_score = util.pytorch_cos_sim(ans_embedding, actual_ans_embedding)[0][0].item()

    return {
        'question': df.iloc[question_index]['question'],
        'actual_ans': actual_ans,
        'similarity_score': similarity_score
    }

# Display the list of questions
print("Following are the questions:\n")
for idx, question in enumerate(df['question'], 1):
    print(f"{idx}. {question}")

# Ask the user to choose a question
n = int(input("Which question number do you want to answer? "))

# Get the user's answer
ans = input("Enter your answer: ")

# Get the suggestion for the selected question
suggestion = suggest_section_for_selected_question(ans, df, n-1)

# Display the result
if suggestion:
    print(f"\nQuestion number: {n}")
    print(f"Question: {suggestion['question']}")
    print(f"\nActual Answer: {suggestion['actual_ans']}")
    print(f"Your Answer: {ans}")
    print(f"\n{round(suggestion['similarity_score']*100)}% your answer is correct.")


Following are the questions:

1. What is Python? List some popular applications of Python in the world of technology.
2. What are the benefits of using Python language as a tool in the present scenario?
3. What is the difference between a Mutable datatype and an Immutable data type?
4. What is the difference between a Set and Dictionary?
5. What is a pass in Python?
6. Difference between for loop and while loop in Python
7. Not Mentioned

Question number: 1
Question: What is Python? List some popular applications of Python in the world of technology.

Actual Answer: Python is a widely-used general-purpose, high-level programming language. It was created by Guido van Rossum in 1991 and further developed by the Python Software Foundation. It was designed with an emphasis on code readability, and its syntax allows programmers to express their concepts in fewer lines of code.
Your Answer: it is so easy to use

26% your answer is correct.


In [ ]:
import seven as ap
ap.audio_to_text("C:/Users/amans/Desktop/audio_to_text_audio_file.wav")